In [1]:
from dotenv import load_dotenv
load_dotenv()

import getpass, os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")


In [2]:
# Low temperature for reasoning
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0
)


In [3]:
question = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many does he have now?"
prompt_standard = f"Answer this question: {question}"

print("---- STANDARD PROMPT ----")
print(llm.invoke(prompt_standard).content)


---- STANDARD PROMPT ----
To find out how many tennis balls Roger has now, we need to add the initial number of tennis balls he had (5) to the number of tennis balls he bought (2 cans * 3 tennis balls per can).

2 cans * 3 tennis balls per can = 6 tennis balls

Now, let's add the initial number of tennis balls (5) to the number of tennis balls he bought (6):

5 + 6 = 11

So, Roger now has 11 tennis balls.


In [4]:
prompt_cot = f"Answer this question. Let's think step by step. {question}"

print("---- CHAIN OF THOUGHT ----")
print(llm.invoke(prompt_cot).content)


---- CHAIN OF THOUGHT ----
To find out how many tennis balls Roger has now, we need to follow these steps:

1. Roger already has 5 tennis balls.
2. He buys 2 more cans of tennis balls. Each can has 3 tennis balls, so he buys 2 x 3 = 6 more tennis balls.
3. Now, we add the tennis balls he already had (5) to the new tennis balls he bought (6). 5 + 6 = 11

So, Roger now has 11 tennis balls.


Observation

When asked directly, the model may skip reasoning.
When asked to “think step by step”, it generates intermediate reasoning tokens, improving accuracy.

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

problem = "How can I get my 5-year-old to eat vegetables?"

prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Give one unique, creative solution. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three proposed solutions for: '{problem}'
    
    1: {sol1}
    2: {sol2}
    3: {sol3}
    
    Act as a Child Psychologist.
    Pick the most sustainable solution (no bribery) and explain why.
    """
)


In [6]:
tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]})
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("---- TREE OF THOUGHTS RESULT ----")
print(tot_chain.invoke(problem))


---- TREE OF THOUGHTS RESULT ----
As a child psychologist, I would recommend Solution 2: "Veggie Gardening" as the most sustainable solution to encourage a 5-year-old to eat vegetables. Here's why:

1. **Ownership and Responsibility**: By involving your child in the process of growing their own vegetables, you're giving them a sense of ownership and responsibility. This can lead to a deeper appreciation for the food they eat and a willingness to try new vegetables.
2. **Hands-on Learning**: Gardening and cooking are hands-on activities that allow children to learn through experience. This approach can help your child develop essential life skills, such as nurturing, patience, and problem-solving.
3. **Natural Curiosity**: Children are naturally curious, and gardening can spark their interest in the natural world. By involving them in the process, you're encouraging their curiosity and sense of wonder.
4. **No Bribery**: Unlike the other solutions, "Veggie Gardening" doesn't rely on bri

Why ToT works

Instead of one answer, the model:

Explores multiple solutions

Evaluates them

Selects the best one

This mimics human decision making.

In [7]:
prompt_draft = ChatPromptTemplate.from_template(
    "Write a 1-sentence movie plot about: {topic}. Genre: {genre}."
)

drafts = RunnableParallel(
    draft_scifi=prompt_draft.partial(genre="Sci-Fi") | llm | StrOutputParser(),
    draft_romance=prompt_draft.partial(genre="Romance") | llm | StrOutputParser(),
    draft_horror=prompt_draft.partial(genre="Horror") | llm | StrOutputParser(),
)


In [8]:
prompt_combine = ChatPromptTemplate.from_template(
    """
    I have three movie ideas for the topic '{topic}':
    1. Sci-Fi: {draft_scifi}
    2. Romance: {draft_romance}
    3. Horror: {draft_horror}
    
    Create a Mega-Movie combining the TECHNOLOGY of Sci-Fi,
    PASSION of Romance, and FEAR of Horror.
    
    Write one paragraph.
    """
)


In [9]:
got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]})
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("---- GRAPH OF THOUGHTS RESULT ----")
print(got_chain.invoke("Time Travel"))


---- GRAPH OF THOUGHTS RESULT ----
In "Echoes of Eternity," a brilliant physicist, Emma, discovers a way to manipulate time using a revolutionary technology that allows her to traverse the fabric of reality. As she experiments with her invention, she finds herself reliving the same pivotal moment in her past over and over, each time with the chance to rekindle a lost love, Jack, who was tragically taken from her in a car accident. However, their rekindled romance is short-lived, as a malevolent entity from the past, a vengeful spirit of a woman who was wronged by Jack's ancestors, begins to hunt them down, forcing Emma and Jack to relive the same terrifying night over and over in a desperate bid to survive. As they navigate the complex web of time, they must also confront the dark secrets of their own pasts and the true cost of their love, all while trying to prevent the catastrophic future that Emma's actions have inadvertently created.


| Method            | Structure                 | Best Use           |
| ----------------- | ------------------------- | ------------------ |
| Simple Prompt     | Input → Output            | Facts, summaries   |
| Chain of Thought  | Input → Steps → Output    | Math, logic        |
| Tree of Thoughts  | Input → Branch → Judge    | Decision making    |
| Graph of Thoughts | Branch → Combine → Output | Creative synthesis |
